# CISC455/851: Adaptive Evolutionary Traffic Optimization
---
## 1. Professional Infrastructure and Logic
This notebook implements a modular Evolutionary Algorithm (EA) framework that optimizes traffic light timings. 

It scales automatically based on grid size and offers a headless mode for high-speed training.

### Assignment Adherence:
* **Varying Distances**: Intersections are spaced non-uniformly to provide realistic complexity.
* **Gatekeeping Logic**: Vehicles cannot enter an intersection if the road ahead is full (collision avoidance).
* **Bidirectional Flow**: Support for North, South, East, and West traffic through multiple parallel arrays.
* **Evolutionary Optimization**: Utilizes Block-Level Crossover, Tournament Selection, Mutation, and Elitism to minimize total travel time and idling.

## 2. Configuration Dashboard
Adjust these parameters to scale the environment, tune the Evolutionary Algorithm, or toggle the simulation window.

In [1]:
import numpy as np
import random
import pickle
import copy
import pygame
import sys

# --- Environment Scaling ---
GRID_SIZE = 4            # Scales to NxN intersections automatically
HEADLESS_MODE = False    # Set to True to turn off the pygame window for faster CPU training
ROAD_LEN = 1000          # Total length of each road channel (in meters)
SPAWN_RATE = 0.12        # Probability of a vehicle spawning per second (approx 12% chance)

# --- EA Parameters (Tuning Knobs) ---
POP_SIZE = 12            # Number of unique timing strategies per generation
GENS = 15                # Number of evolutionary cycles (generations)
MUTATION_RATE = 0.3      # Probability of a random block changing (promotes Exploration)
CROSSOVER_RATE = 0.8     # Probability of combining two parents (promotes Exploitation)
TOURNAMENT_K = 3         # Size of tournament selection pool (higher = more greedy selection)

# Generate Varied Intersections based on GRID_SIZE
# This ensures distance between intersections is not uniform, adding complexity.
INTERSECTIONS = np.linspace(100, 900, GRID_SIZE) + np.random.randint(-20, 20, GRID_SIZE)
INTERSECTIONS = sorted(INTERSECTIONS.tolist())

pygame 2.6.1 (SDL 2.28.4, Python 3.12.10)
Hello from the pygame community. https://www.pygame.org/contribute.html


## 3. Vehicle Physics and Genotype Logic
Defines the agents (vehicles) and the genetic representation of the traffic lights.

* **Genotype Constraint**: A `TimingBlock` enforces a strict sequence structure. Whenever a light transitions from Green to Red, it is forced to undergo a 6-second Yellow phase and a 3-second All-Red safety phase. This guarantees all generated chromosomes represent valid, legal traffic states.

In [2]:
class Vehicle:
    def __init__(self, vid, vtype, axis, direction, channel):
        self.id = vid
        self.length = 1 if vtype == 'car' else 3
        self.axis = axis           # 'h' (horizontal) or 'v' (vertical)
        self.direction = direction # 1 (Forward: 0->1000) or -1 (Backward: 1000->0)
        self.pos = 0 if direction == 1 else ROAD_LEN
        self.channel = channel     # The specific road coordinate the vehicle is on
        
        self.speed = 10            # Spawn at top speed (10 m/s)
        self.accel = 2 if vtype == 'car' else 1
        self.idling_time = 0
        self.travel_time = 0
        self.finished = False

    def step(self, dist_to_front):
        if self.finished: 
            return
            
        # Acceleration & Braking Logic: Maintain a 2m safety buffer
        if dist_to_front <= self.speed + 2:
            self.speed = max(0, dist_to_front - 2)
            if self.speed == 0: 
                self.idling_time += 1
        else:
            self.speed = min(10, self.speed + self.accel)
        
        self.pos += (self.speed * self.direction)
        self.travel_time += 1
        
        # Exit condition: Mark as finished once the boundary is crossed
        if (self.direction == 1 and self.pos >= ROAD_LEN) or (self.direction == -1 and self.pos <= 0):
            self.finished = True

class TimingBlock:
    def __init__(self, g_ns, g_ew):
        # EA Genotype Representation:
        # 0: NS Green | 1: NS Yellow | 2: EW Green | 3: EW Yellow | 4: All Red
        # Enforces '111111444' and '333333444' mandatory safety sequences
        self.seq = ([0]*g_ns) + [1,1,1,1,1,1,4,4,4] + ([2]*g_ew) + [3,3,3,3,3,3,4,4,4]

## 4. Adaptive Simulation Engine and Fitness Evaluation
Handles rendering and calculates the fitness of each individual chromosome.

### The Fitness Function (Minimization Problem)
The EA aims to find the lowest possible score. 
1. **Base Score**: Sum of travel time + (idling time * 2). Idling is penalized twice as heavily to discourage gridlock.
2. **Completion Gradient Penalty**: Vehicles that fail to exit the grid add a massive 5000-point penalty PLUS their remaining distance to the exit. This provides a clear mathematical gradient for the EA to follow even if zero cars finish in early generations.

In [3]:
def render_adaptive(screen, vehicles, light, info):
    """ Renders the grid network and simulation HUD. """
    for event in pygame.event.get():
        if event.type == pygame.QUIT: 
            pygame.quit()
            sys.exit()
            
    screen.fill((230, 230, 230))
    
    # Draw the road infrastructure
    for r in INTERSECTIONS:
        pygame.draw.rect(screen, (50, 50, 50), (0, r * 0.8 - 20, 800, 40))
        pygame.draw.rect(screen, (50, 50, 50), (r * 0.8 - 20, 0, 40, 800))
    
    # Draw the traffic lights
    c = {0: (0, 255, 0), 1: (255, 255, 0), 2: (0, 100, 0), 3: (100, 100, 0), 4: (255, 0, 0)}
    for rx in INTERSECTIONS:
        for ry in INTERSECTIONS:
            pygame.draw.circle(screen, c.get(light, (255, 0, 0)), (int(rx * 0.8), int(ry * 0.8)), 14)
            
    # Draw active vehicles
    for v in vehicles:
        if v.finished: 
            continue
        col = (0, 100, 255) if v.length == 1 else (255, 140, 0)
        s_pos = v.pos * 0.8
        if v.axis == 'h':
            lane_off = -15 if v.direction == 1 else 5
            pygame.draw.rect(screen, col, (s_pos, v.channel * 0.8 + lane_off, v.length * 8, 12))
        else:
            lane_off = -15 if v.direction == 1 else 5
            pygame.draw.rect(screen, col, (v.channel * 0.8 + lane_off, s_pos, 12, v.length * 8))

    # Draw HUD and metrics
    pygame.draw.rect(screen, (255, 255, 255), (0, 800, 800, 50))
    font = pygame.font.SysFont("Consolas", 16)
    gen_txt = font.render(f"GEN: {info['gen']} | SEED: {info['seed']}", True, (0, 0, 0))
    time_txt = font.render(f"SIM TIME: {info['t']}s / 3600s", True, (0, 0, 0))
    count_txt = font.render(f"ACTIVE VEHICLES: {info['active']}", True, (0, 0, 0))
    screen.blit(gen_txt, (20, 810))
    screen.blit(time_txt, (250, 810))
    screen.blit(count_txt, (550, 810))
    pygame.display.flip()

def evaluate(genes, seed, screen=None, clock=None, gen_idx=0):
    """ Simulates 1 hour of traffic and returns the fitness score of the given chromosome. """
    random.seed(seed)
    vehicles = []
    active_vehicles = [] # Performance optimization: Limits O(N^2) collision checks to active cars only
    vid = 0
    
    # Unpack TimingBlocks into a flat 3600-length array of states
    full_genes_list = []
    for b in genes:
        full_genes_list.extend(b.seq)
    if len(full_genes_list) < 3600:
        full_genes_list.extend([4] * (3600 - len(full_genes_list)))
    full_genes = np.array(full_genes_list[:3600])

    for t in range(3600):
        # Continuous Spawning Logic
        if random.random() < SPAWN_RATE:
            ax, dr, ch = random.choice(['h','v']), random.choice([1,-1]), random.choice(INTERSECTIONS)
            
            # Gatekeeping: Only spawn if the entrance is clear of other vehicles
            spawn_pos = 0 if dr == 1 else ROAD_LEN
            if all(abs(v.pos - spawn_pos) > 20 for v in active_vehicles if v.axis == ax and v.channel == ch and v.direction == dr):
                new_v = Vehicle(vid, random.choice(['car','bus']), ax, dr, ch)
                vehicles.append(new_v)
                active_vehicles.append(new_v)
                vid += 1
        
        light = full_genes[t]
        if screen and t % 10 == 0:
            info = {'gen': gen_idx, 'seed': seed, 't': t, 'active': len(active_vehicles)}
            render_adaptive(screen, vehicles, light, info)
            clock.tick(120)
        
        # Physics and Collision Step
        next_active = []
        for v in active_vehicles:
            dist = float('inf') 
            
            # 1. Intersection Stopping Logic
            valid_stops = []
            for stop in INTERSECTIONS:
                # Stop line is offset 12 meters before the intersection coordinate to prevent overlap
                stop_line = stop - 12 if v.direction == 1 else stop + 12
                if (v.direction == 1 and v.pos < stop_line) or (v.direction == -1 and v.pos > stop_line):
                    valid_stops.append(stop_line)
            
            if valid_stops:
                closest_stop = min(valid_stops, key=lambda x: abs(x - v.pos))
                green = (light == 0 or light == 1) if v.axis == 'v' else (light == 2 or light == 3)
                if not green:
                    dist = min(dist, abs(closest_stop - v.pos))
            
            # 2. Vehicle-to-Vehicle Collision Avoidance
            for other in active_vehicles:
                # Only check cars in the exact same lane traveling in the same direction
                if other.id != v.id and other.axis == v.axis and other.channel == v.channel and other.direction == v.direction:
                    # Only react to cars physically ahead of this vehicle
                    if (v.direction == 1 and other.pos > v.pos) or (v.direction == -1 and other.pos < v.pos):
                        dist = min(dist, abs(other.pos - v.pos) - other.length)
            
            v.step(dist)
            
            # Garbage collection: drop finished vehicles from active memory
            if not v.finished: 
                next_active.append(v)
            
        active_vehicles = next_active

    # --- Fitness Calculation ---
    finished = [v for v in vehicles if v.finished]
    score = sum([v.travel_time + (v.idling_time * 2) for v in finished])
    penalty = sum([5000 + (abs((ROAD_LEN if v.direction == 1 else 0) - v.pos)) for v in vehicles if not v.finished])
    
    # Normalize by total spawned vehicles to prevent bias against high-spawn scenarios
    return (score + penalty) / max(len(vehicles), 1)

## 5. Evolutionary Logic (Selection & Variation)
This section dictates how the population reproduces and evolves over generations.

1. **Tournament Selection**: Selects `k` random individuals and pits them against each other, returning the one with the best (lowest) fitness. This applies positive selection pressure while avoiding the premature convergence caused by purely greedy selection.
2. **Block-Level Crossover**: Combines genetic traits of two parents. By splitting at the `TimingBlock` level rather than the individual second level, the algorithm guarantees that no safety transition periods (yellow/red phases) are corrupted during reproduction.

In [4]:
def tournament_selection(population, fitness_scores, k=3):
    """ Selects a parent by holding a tournament among k random individuals. """
    selected_indices = random.sample(range(len(population)), k)
    best_idx = selected_indices[0]
    for idx in selected_indices[1:]:
        if fitness_scores[idx] < fitness_scores[best_idx]:
            best_idx = idx
    return population[best_idx]

def crossover(parent1, parent2):
    """ Combines two parent chromosomes using single-point block-level crossover. """
    if random.random() < CROSSOVER_RATE:
        split = random.randint(1, len(parent1) - 1)
        return parent1[:split] + parent2[split:]
    return copy.deepcopy(parent1)

## 6. Training Execution
The main loop that drives the simulation. It evaluates a baseline (a fixed 30s Green/30s Red loop) to prove whether the EA is genuinely learning to improve traffic flow, or simply failing. Utilizes **Elitism** to preserve the absolute best strategy across generations.

In [5]:
screen, clock = None, None
if not HEADLESS_MODE: 
    pygame.init()
    screen = pygame.display.set_mode((800, 850))
    clock = pygame.time.Clock()

# Initialize starting population with random timing blocks
pop = [[TimingBlock(random.randint(15,50), random.randint(15,50)) for _ in range(40)] for _ in range(POP_SIZE)]
baseline_genes = [TimingBlock(30, 30) for _ in range(60)]

for gen in range(GENS):
    # Seed shifts each generation to prevent overfitting to a single traffic pattern
    seed = 3000 + gen
    
    sys.stdout.write(f"\r[Gen {gen}] Evaluating Baseline...")
    sys.stdout.flush()
    base_score = evaluate(baseline_genes, seed)
    
    # Evaluate all individuals in the current population
    scores = []
    for i, ind in enumerate(pop):
        sys.stdout.write(f"\r[Gen {gen}] Evaluating Strategy {i+1}/{POP_SIZE}...")
        sys.stdout.flush()
        # Only render the first individual to save visualizer time
        scores.append(evaluate(ind, seed, (screen if i==0 and not HEADLESS_MODE else None), clock, gen))
    
    best_idx = np.argmin(scores)
    diff = base_score - scores[best_idx]
    sys.stdout.write(f"\rGen {gen} | Seed: {seed} | EA Best: {scores[best_idx]:.1f} | Base: {base_score:.1f} | Diff: {diff:+.1f}       \n")
    
    # --- Reproduction Phase ---
    # Elitism: Automatically copy the single best individual to the next generation unaltered
    new_pop = [pop[best_idx]] 
    
    # Fill the remainder of the new generation
    while len(new_pop) < POP_SIZE:
        p1 = tournament_selection(pop, scores, k=TOURNAMENT_K)
        p2 = tournament_selection(pop, scores, k=TOURNAMENT_K)
        child = crossover(p1, p2)
        
        # Mutation Phase: Randomly replace one block to introduce new genetic traits
        if random.random() < MUTATION_RATE:
            child[random.randint(0, 39)] = TimingBlock(random.randint(10, 80), random.randint(10, 80))
            
        new_pop.append(child)
        
    pop = new_pop

if not HEADLESS_MODE: 
    pygame.quit()
    
# Serialize and save the globally best individual for Validation plotting
with open('best_timing.pkl', 'wb') as f: 
    pickle.dump(pop[0], f)

print("\nTraining Complete. Model saved as best_timing.pkl")

Gen 0 | Seed: 3000 | EA Best: 585.2 | Base: 392.6 | Diff: -192.6       
Gen 1 | Seed: 3001 | EA Best: 575.2 | Base: 384.6 | Diff: -190.6       
Gen 2 | Seed: 3002 | EA Best: 485.3 | Base: 402.9 | Diff: -82.4       
Gen 3 | Seed: 3003 | EA Best: 513.3 | Base: 390.8 | Diff: -122.5       
Gen 4 | Seed: 3004 | EA Best: 560.1 | Base: 474.7 | Diff: -85.3       
Gen 5 | Seed: 3005 | EA Best: 565.1 | Base: 465.3 | Diff: -99.9       
Gen 6 | Seed: 3006 | EA Best: 489.8 | Base: 387.5 | Diff: -102.3       
Gen 7 | Seed: 3007 | EA Best: 532.6 | Base: 433.1 | Diff: -99.5       
Gen 8 | Seed: 3008 | EA Best: 561.6 | Base: 468.5 | Diff: -93.1       
Gen 9 | Seed: 3009 | EA Best: 558.0 | Base: 455.3 | Diff: -102.7       
Gen 10 | Seed: 3010 | EA Best: 646.7 | Base: 527.2 | Diff: -119.5       
Gen 11 | Seed: 3011 | EA Best: 485.2 | Base: 391.3 | Diff: -93.9       
Gen 12 | Seed: 3012 | EA Best: 496.2 | Base: 417.1 | Diff: -79.1       
Gen 13 | Seed: 3013 | EA Best: 519.4 | Base: 435.9 | Diff: -83.5    

### Output Artifact: `best_timing.pkl`

Upon successful completion of the training loop, the script serializes the most successful individual from the final generation and saves it locally as `best_timing.pkl` using Python's `pickle` library.

#### What is inside this file?
It contains the **Genotype** of the winning traffic strategy. Specifically, it is a Python list containing 40 `TimingBlock` objects. Each `TimingBlock` encapsulates a rigid sequence of integers (representing green, yellow, and red states) that dictate the flow of the 4x4 intersection over a given interval.

#### How to use it:
This file is meant to be loaded by the **Validation Notebook**. By decoupling the training process from the evaluation process, you can train a highly optimized model in 'Headless Mode' for hundreds of generations in the background, save the resulting `.pkl` file, and then load it into a separate visualizer later for the final presentation and throughput analysis.